# Helping turn website to data

## Publications

HTML data to JSON

In [6]:
from bs4 import BeautifulSoup, Tag
import json
import re

def get_text_without_links(tag):
    """Extract text from a tag, removing link elements but keeping surrounding text clean."""
    result = tag
    # Clone to avoid modifying original
    clone = BeautifulSoup(str(tag), "html.parser").find(tag.name)
    for a in clone.find_all("a"):
        a.decompose()
    # Also remove span tags (e.g. hashtag labels)
    for span in clone.find_all("span"):
        span.decompose()
    # Clean up artifacts like brackets around removed links: [ ] or []
    text = clone.get_text(separator=" ")
    text = re.sub(r'\[\s*\]', '', text)       # remove empty brackets
    text = re.sub(r'\s{2,}', ' ', text)       # collapse whitespace
    text = re.sub(r'\s([,.])', r'\1', text)   # fix space before punctuation
    return text.strip()

# def get_links(tag):
#     """Extract href + link text from all <a> tags that look like DOI/PDF/URL links."""
#     links = []
#     for a in tag.find_all("a"):
#         href = a.get("href")
#         text = a.get_text(strip=True)
#         # Skip anchor/named links (e.g. <a name="..."> or <a id="...">)
#         if href and not href.startswith("#"):
#             links.append({"text": text, "url": href})
#     return links

def get_pdf_doi(tag, pdf_doi="pdf"):
    """Extract href + link text from all <a> tags that look like DOI/PDF/URL links."""
    _pdf_doi_link = ""
    for a in tag.find_all("a"):
        href = a.get("href")
        text = a.get_text(strip=True)
        if text.lower()!=pdf_doi:
            continue
        # Skip anchor/named links (e.g. <a name="..."> or <a id="...">)
        if href and not href.startswith("#"):
            _pdf_doi_link = href
    return _pdf_doi_link

def get_tags(tag):
    """Extract hashtag labels from <span class="tag-*"> elements."""
    tags = []
    for span in tag.find_all("span"):
        text = span.get_text(strip=True)
        if text:
            tags.append(text.replace("#",""))
    return tags

def is_empty_div(div):
    """Check if a grid_1 div has no meaningful text content."""
    text = div.get_text(strip=True).replace('\xa0', '').strip()
    return text == ''

import re

def get_title(text):
    """Extract title: text after first (YEAR). up to the next period."""
    match = re.search(r'\(\d{4}\)\.*\s*(.+?)\.', text)
    return match.group(1).strip() if match else None

def parse_publications(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")

    # Grab all divs with class grid_1 or grid_10 in document order
    divs = soup.find_all("div", class_=re.compile(r"^grid_(1|10)$"))

    results = []
    current_type = None
    i = 0

    while i < len(divs):
        # Expect a triple: grid_1 (type), grid_1 (year), grid_10 (publications)
        if i + 2 >= len(divs):
            break

        type_div  = divs[i]
        year_div  = divs[i + 1]
        pubs_div  = divs[i + 2]

        # Validate the triple structure
        if ("grid_1"  not in type_div.get("class", []) or
            "grid_1"  not in year_div.get("class", []) or
            "grid_10" not in pubs_div.get("class", [])):
            i += 1
            continue

        # --- Type ---
        if not is_empty_div(type_div):
            # Strip any anchor tags used as named anchors, get clean text
            raw_type = type_div.get_text(strip=True).replace('\xa0', '').strip()
            if raw_type:
                current_type = raw_type
        # If empty, current_type carries forward from previous triple

        # --- Year ---
        year = year_div.get_text(strip=True).replace('\xa0', '').strip()

        # --- Publications (one entry per <p>) ---
        for p in pubs_div.find_all("p"):
            pub_text = get_text_without_links(p)
            title = get_title(pub_text)
            pdf_link = get_pdf_doi(p, "pdf")
            doi_link = get_pdf_doi(p, "doi")
            tags = get_tags(p)

            if pub_text:  # skip empty paragraphs
                results.append({
                    "type":  current_type.lower(),
                    "year":  year,
                    "title": title,
                    "text":  pub_text,
                    **({"pdf": pdf_link} if pdf_link else {}),
                    **({"doi": doi_link} if doi_link else {}),
                    **({"tags": tags}if tags else {}),
                })
                # if pdf_link:
                #     entry["pdf"] = pdf_link
                # if doi_link:
                #     entry["doi"] = doi_link
                
                # results.append(entry)
        i += 3  # advance past the processed triple

    return results


# ── Demo ──────────────────────────────────────────────────────────────────────
html = """
<!-- Conference 2019 -->
<div class="grid_1">&nbsp;</div>
<div class="grid_1"><p>2019</p></div>
<div class="grid_10">
	<p>Yudelson, M., Rosen, Y., Polyak, S., &amp; de la Torre, J. (2019) <strong>Leveraging Skill Hierarchy for Multi-Level Modeling with Elo Rating System</strong>. In Proceedings of the Sixth ACM Conference on Learning at Scale (L@S 2019), Chicagom IL, USA, June 24-25, 2019, (32:1&mdash;32:4). ACM, New York, NY, USA. [<a href="pdf/LaS2019_YudelsonRPdlT_WiP.pdf" target="_target">PDF</a>] [<a href="https://doi.org/10.1145/3330430.3333645" target="_target">DOI</a>]&nbsp;<span class="tag-red">#Elo</span>&nbsp;<span class="tag-green">#BigData&Scale</span></p>
	<p>Yudelson, M. (2019) <strong>Elo, I Love You Won’t You Tell Me Your K</strong>. In: M. Scheffel, J. Broisin, V. Pammer-Schindler, A. Ioannou, &amp; J. Schneider (Eds.) Proceedings of 14th European Conference on Technology Enhanced Learning (EC-TEL 2019), Delft, The Netherlands, September 16-19, 2019, (213&mdash;223). Springer,  [<a href="pdf/ECTEL2019_Yudelson_Elo.pdf" target="_target">PDF</a>] [<a href="https://doi.org/10.1007/978-3-030-29736-7_16" target="_target">DOI</a>]&nbsp;<span class="tag-red">#Elo</span></p>
</div> 
<div class="grid_1"><p><a id="Journal"></a>Journal</p></div>
<div class="grid_1"><p>2019</p></div>
<div class="grid_10">
    <p><a name="CPAtHLaAs"></a>Von Davier, A. A., Deonovic, B. E., Yudelson, M., Polyak, S., &amp; Woo, A. (2019). <strong>Computational Psychometrics Approach to Holistic Learning and Assessment Systems</strong>. <em>Frontiers in Education</em>, (4), 69. [<a href="https://doi.org/10.3389/feduc.2019.00069" target="_target">DOI</a>]</p>
    <p>Von Davier, A. A., Wong, P. C., Polyak, S., &amp; Yudelson, M. (2019). <strong>The Argument for a "Data Cube" for Large-Scale Psychometric Data</strong>. <em>Frontiers in Education</em> 4, 71. [<a href="https://doi.org/10.3389/feduc.2019.00071" target="_target">DOI</a>]&nbsp;<span class="tag-green">#BigData&Scale</span></p>
</div>
<div class="grid_1"><p>&nbsp;</div>
<div class="grid_1"><p>2018</p></div>
<div class="grid_10">
    <p>Deonovic, B., Yudelson, M., Bolsinova, M., Attali, M., &amp; Maris, G. (2018). <strong>Learning meets assessment.</strong>. <em>Behaviormetrika</em> 45(2), 457&mdash;474. [<a href="https://doi.org/10.1007/s41237-018-0070-z" target="_target">DOI</a>]</p>
</div>
<div class="grid_1">&nbsp;</div>
<div class="grid_1"><p>2016</p></div>
<div class="grid_10">
    <p>Koedinger, K., Yudelson, M., &amp; Pavlik, P. (2016). <strong>Testing Theories of Transfer</strong>. <em>Topics in Cognitive Science, 8</em> (3), 589&mdash;609. [<a href="http://dx.doi.org/10.1111/tops.12208" target="_target">DOI</a>]</p>
</div>
<!-- Journal 2018 -->
<div class="grid_1"><p>&nbsp;</div>
<div class="grid_1"><p>2018</p></div>
<div class="grid_10">
		<p>Deonovic, B., Yudelson, M., Bolsinova, M., Attali, M., &amp; Maris, G. (2018). <strong>Learning meets assessment. On the relation between item response theory and Bayesian knowledge tracing</strong>. <em>Behaviormetrika, TBD</em> 45(2), 457&mdash;474. [<a href="https://doi.org/10.1007/s41237-018-0070-z" target="_target">DOI</a>]&nbsp;<span class="tag-blue">#HMM</span></p>
</div>
"""

# publications = parse_publications(html)
# print(json.dumps(publications, indent=4, ensure_ascii=False))
with open("publification.html", "r", encoding="utf-8") as f:
    html = f.read()

publications_json = parse_publications(html)

with open("publications.json", "w") as f:
    f.write(json.dumps(publications_json, indent=4, ensure_ascii=False))


## Vitae

Reverse list

In [8]:
import json
with open("vitae.json", "r") as f:
    jsn = json.load(f)

vi = jsn["en"]
jsn["em"] = vi.reverse()

with open("vitae.json", "w") as f:
    f.write(json.dumps(jsn, indent=4, ensure_ascii=False))
